# Addis Ababa University
# Advanced programming for Bioinformatics 
# Module 3 part 2: Biopython for Bioinformatics and Genomics

**Primary reference:** Official Biopython Tutorial and Cookbook  
https://biopython.org/docs/latest/Tutorial/

It focuses only on Biopython-based biological sequence analysis and related workflows.

## Learning Outcomes

By the end of this notebook, students should be able to:

1. Create and manipulate biological sequences using `Bio.Seq`
2. Use `SeqRecord` objects to store sequence metadata
3. Read and write FASTA files using `SeqIO`
4. Calculate sequence statistics such as length, base composition, and GC content
5. Perform transcription, translation, and reverse-complement operations
6. Parse GenBank-style annotations conceptually
7. Access NCBI Entrez records programmatically
8. Perform pairwise sequence alignment using Biopython
9. Read multiple sequence alignments using `AlignIO`
10. Read and visualize phylogenetic trees using `Phylo`
11. Build a mini Biopython-based sequence analysis pipeline

## 0. Setup/installation

Run the cell below if Biopython is not installed.

In [ ]:
# Uncomment and run if Biopython is not installed
# !pip install biopython matplotlib pandas logomaker

In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO, AlignIO, Align, Entrez, Phylo
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from io import StringIO

print("Biopython core modules imported successfully.")

# Define GC content function early (solution for Exercise 1)
def calculate_gc(sequence):
    """Calculate GC content of a DNA sequence as percentage."""
    seq_str = str(sequence).upper()
    if len(seq_str) == 0:
        return 0.0
    gc_count = seq_str.count('G') + seq_str.count('C')
    return (gc_count / len(seq_str)) * 100

# Part 1 — Biological Sequences with `Bio.Seq`

Biopython represents biological sequences using the `Seq` object. A `Seq` object behaves like a Python string but includes biological methods such as transcription, translation, and reverse-complement generation.

In [ ]:
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
print("DNA sequence:", dna)
print("Length:", len(dna))
print("Type:", type(dna))
print("A:", dna.count("A"))
print("T:", dna.count("T"))
print("G:", dna.count("G"))
print("C:", dna.count("C"))
print(f"GC content: {calculate_gc(dna):.2f}%")

## Classroom Question

Look at the sequence below:

`ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG`

What tells you that this is probably a DNA sequence and not an RNA sequence?

**Solution:** The presence of thymine (T) instead of uracil (U) indicates it is DNA.

## Exercise 1 — Write a GC Content Function (already defined above)

The function `calculate_gc()` is already implemented.

In [ ]:
# Test the function
print(calculate_gc("ATGC"))  # Expected 50.0
print(calculate_gc("GGCC"))  # Expected 100.0

# Part 2 — Reverse Complement, Transcription, and Translation

In [ ]:
print("Original DNA:        ", dna)
print("Complement:          ", dna.complement())
print("Reverse complement:  ", dna.reverse_complement())
rna = dna.transcribe()
print("RNA:", rna)
protein = dna.translate()
print("Protein:", protein)
print("Standard translation:", dna.translate())
print("Stop at first stop codon:", dna.translate(to_stop=True))

## Exercise 2 — Translate Your Own DNA Sequence

Create a DNA sequence that starts with `ATG`, translate it, and explain the resulting amino acid sequence.

In [ ]:
# Solution
student_dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")
print("My DNA:", student_dna)
print("Protein:", student_dna.translate())
print("Explanation: The DNA sequence codes for the amino acid sequence MAIVMGR* where * represents a stop codon.")

# Part 3 — SeqRecord Objects

In [ ]:
record = SeqRecord(
    dna,
    id="Example_001",
    name="ExampleGene",
    description="Example DNA sequence for MSc Biopython teaching"
)
print(record)

## Exercise 3 — Create a SeqRecord

Create your own `SeqRecord` with an ID, name, description, and a DNA sequence.

In [ ]:
# Solution
my_seq = Seq("ATGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG")
my_record = SeqRecord(
    my_seq,
    id="MySeq_001",
    name="MyGene",
    description="My own DNA sequence for practice"
)
print(my_record)

# Part 4 — Reading and Writing FASTA Files with SeqIO

In [ ]:
fasta_text = ">seq1 Example sequence 1\nATGCGTACGTAGCTAGCTAG\n>seq2 Example sequence 2\nATGCCCGGGTTTAAACCCGGG\n>seq3 Example sequence 3\nATATATATATATATATATATA\n"
with open("example_sequences.fasta", "w") as handle:
    handle.write(fasta_text)
print("Example FASTA file created.")

In [ ]:
for record in SeqIO.parse("example_sequences.fasta", "fasta"):
    print("ID:", record.id)
    print("Description:", record.description)
    print("Sequence:", record.seq)
    print("Length:", len(record.seq))
    print("GC content:", f"{calculate_gc(record.seq):.2f}%")
    print("-" * 40)

## Exercise 4 — FASTA Summary Table

Read the example FASTA file and create a summary table with sequence ID, length, GC content, and base counts.

In [ ]:
# Solution
summary = []
for record in SeqIO.parse("example_sequences.fasta", "fasta"):
    seq_str = str(record.seq).upper()
    summary.append({
        "id": record.id,
        "length": len(seq_str),
        "GC%": calculate_gc(seq_str),
        "A": seq_str.count('A'),
        "T": seq_str.count('T'),
        "G": seq_str.count('G'),
        "C": seq_str.count('C')
    })
df_summary = pd.DataFrame(summary)
df_summary

# Part 5 — Simple Visualization of Sequence Composition

In [ ]:
record = next(SeqIO.parse("example_sequences.fasta", "fasta"))
seq = str(record.seq)
bases = ["A", "T", "G", "C"]
counts = [seq.count(base) for base in bases]
plt.figure(figsize=(6,4))
plt.bar(bases, counts)
plt.xlabel("Base")
plt.ylabel("Count")
plt.title(f"Base Composition: {record.id}")
plt.show()

# Part 6 — Writing New FASTA Files

In [ ]:
records = []
for record in SeqIO.parse("example_sequences.fasta", "fasta"):
    new_record = SeqRecord(
        record.seq.reverse_complement(),
        id=record.id + "_reverse_complement",
        description="Reverse complement sequence"
    )
    records.append(new_record)
SeqIO.write(records, "reverse_complements.fasta", "fasta")
print("Reverse complement FASTA file written.")
print(open("reverse_complements.fasta").read())

# Part 7 — GenBank Conceptual Parsing

GenBank records contain sequence plus rich annotations such as genes, CDS regions, products, and references.

In [ ]:
# Example structure only – uncomment to use with real GenBank file
# for record in SeqIO.parse("example.gbk", "genbank"):
#     print(record.id, len(record.seq))
#     for feature in record.features:
#         if feature.type == "CDS":
#             print(feature.location)
#             print(feature.qualifiers.get("gene"))
#             print(feature.qualifiers.get("product"))

## Exercise 5 — Discussion

Why is GenBank more informative than FASTA for genome annotation analysis?

**Solution:** GenBank files include rich metadata: gene names, CDS coordinates, protein products, organism information, and references. FASTA files only store sequence and a single description line. GenBank enables feature-based extraction (e.g., extracting all CDS), while FASTA requires external annotation files. Thus GenBank is essential for detailed genomic analysis.

# Part 8 — Accessing NCBI with Entrez

Biopython provides `Bio.Entrez` for programmatic access to NCBI databases.

**Important:** Always set your email before using Entrez. NCBI requests this for responsible usage.

In [ ]:
# Set your email
Entrez.email = "your_email@example.com"  # Replace with your actual email
print("Entrez email set.")

In [ ]:
# Search example
handle = Entrez.esearch(db="nucleotide", term="Bos taurus mitochondrion complete genome", retmax=5)
results = Entrez.read(handle)
handle.close()
print("IDs:", results["IdList"])

In [ ]:
if results["IdList"]:
    handle = Entrez.efetch(db="nucleotide", id=results["IdList"], rettype="fasta", retmode="text")
    records = list(SeqIO.parse(handle, "fasta"))
    handle.close()
    print(records[0].description)
    print(records[0].seq[:100])

## Exercise 6 — Entrez Search Design

Design Entrez search queries for:

1. Bos taurus mitochondrial genome
2. Capra hircus cytochrome b gene
3. Ovis aries complete mitochondrial genome
4. A gene of your research interest

**Solution:**
1. `"Bos taurus mitochondrion complete genome"`
2. `"Capra hircus cytochrome b"`
3. `"Ovis aries mitochondrion complete genome"`
4. e.g., `"Homo sapiens BRCA1"`

# Part 9 — Pairwise Sequence Alignment

In [ ]:
seq_a = "ATGCGTACGTAG"
seq_b = "ATGCGTTCGTAG"
aligner = Align.PairwiseAligner()
alignments = aligner.align(seq_a, seq_b)
print("Number of alignments:", len(alignments))
print("Best alignment score:", alignments[0].score)
print(alignments[0])

## Exercise 7 — Mutation Interpretation

Align the following sequences and identify the difference:
- Sequence 1: `ATGCCCTAG`
- Sequence 2: `ATGCGCTAG`

In [ ]:
# Solution
seq1 = "ATGCCCTAG"
seq2 = "ATGCGCTAG"
aligner = Align.PairwiseAligner()
alignments = aligner.align(seq1, seq2)
print(alignments[0])
print("Difference: At position 5 (1-indexed), 'C' in seq1 is replaced by 'G' in seq2 (a transition mutation).")

# Part 10 — Multiple Sequence Alignment with AlignIO

In [ ]:
alignment_fasta = ">sample1\nATGCGTACGTAG\n>sample2\nATGCGTTCGTAG\n>sample3\nATGCGTACGCAG\n"
with open("example_alignment.fasta", "w") as handle:
    handle.write(alignment_fasta)
alignment = AlignIO.read("example_alignment.fasta", "fasta")
print(alignment)
print("Number of sequences:", len(alignment))
print("Alignment length:", alignment.get_alignment_length())

In [ ]:
# Sequence logo (requires logomaker)
try:
    import logomaker
    seqs = [str(record.seq) for record in alignment]
    L = len(seqs[0])
    matrix = []
    for i in range(L):
        column = [seq[i] for seq in seqs]
        counts = Counter(column)
        matrix.append(counts)
    df = pd.DataFrame(matrix).fillna(0)
    if "-" in df.columns:
        df = df.drop(columns="-")
    plt.figure(figsize=(15,4))
    logomaker.Logo(df)
    plt.xlabel("Position")
    plt.ylabel("Count")
    plt.title("Sequence Logo")
    plt.show()
except ImportError:
    print("logomaker not installed; skipping logo.")

## Exercise 8 — Alignment Statistics

For each column in the alignment, count how many different bases are present. Which columns are variable?

In [ ]:
# Solution
seqs = [str(record.seq) for record in alignment]
L = len(seqs[0])
variable_columns = []
for i in range(L):
    column = [seq[i] for seq in seqs]
    unique = set(column)
    if len(unique) > 1:
        variable_columns.append((i, unique))
print(f"Variable columns (0-indexed): {len(variable_columns)}")
for idx, bases in variable_columns:
    print(f"Position {idx}: {bases}")

# Part 11 — Phylogenetic Trees with Bio.Phylo

In [ ]:
newick = "((sample1:0.1,sample2:0.2):0.3,sample3:0.4);"
tree = Phylo.read(StringIO(newick), "newick")
Phylo.draw_ascii(tree)

## Exercise 9 — Tree Interpretation

Based on the tree above, which two samples are more closely related? Why?

**Solution:** sample1 and sample2 are more closely related because they share a recent common ancestor (branch length 0.1+0.2) before diverging from sample3 (branch length 0.4).

# Part 12 — Mini Biopython Pipeline

Now we combine several skills into one mini workflow.

The pipeline will:
1. Read a FASTA file
2. Calculate sequence length
3. Calculate GC content
4. Translate sequences that appear to be coding DNA
5. Save a summary table

In [ ]:
def analyze_fasta(input_fasta, output_csv):
    results = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        seq = record.seq
        seq_string = str(seq).upper()
        translated = ""
        if seq_string.startswith("ATG") and len(seq_string) % 3 == 0:
            translated = str(seq.translate(to_stop=False))
        results.append({
            "id": record.id,
            "description": record.description,
            "length": len(seq),
            "gc_content": calculate_gc(seq),
            "starts_with_ATG": seq_string.startswith("ATG"),
            "length_multiple_of_3": len(seq_string) % 3 == 0,
            "translated_sequence": translated
        })
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    return df

pipeline_results = analyze_fasta("example_sequences.fasta", "sequence_summary.csv")
pipeline_results

## Exercise 10 — Improve the Pipeline

Modify the pipeline to also report:
- reverse complement
- number of stop codons in translated protein
- whether the sequence contains ambiguous bases such as N
- AT content
- sequence category: high GC, medium GC, or low GC

In [ ]:
# Improved pipeline
def improved_analyze_fasta(input_fasta, output_csv):
    results = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        seq = record.seq
        seq_string = str(seq).upper()
        rev_comp = str(seq.reverse_complement())
        contains_ambiguous = 'N' in seq_string
        at_content = (seq_string.count('A') + seq_string.count('T')) / len(seq_string) * 100
        gc = calculate_gc(seq)
        if gc >= 60:
            category = "High GC"
        elif gc >= 40:
            category = "Medium GC"
        else:
            category = "Low GC"
        translated = ""
        stop_codons = 0
        if seq_string.startswith("ATG") and len(seq_string) % 3 == 0:
            translated = str(seq.translate(to_stop=False))
            stop_codons = translated.count('*')
        results.append({
            "id": record.id,
            "length": len(seq),
            "GC%": gc,
            "AT%": at_content,
            "category": category,
            "rev_comp": rev_comp,
            "contains_N": contains_ambiguous,
            "starts_with_ATG": seq_string.startswith("ATG"),
            "stop_codons": stop_codons,
            "translated_sequence": translated
        })
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    return df

improved_results = improved_analyze_fasta("example_sequences.fasta", "improved_summary.csv")
improved_results

# Final Challenge Project

Complete one of the following mini-projects using Biopython:

## Option A — Genome FASTA Summary
Search for the smallest genome size data from a public database, download a genome FASTA file and generate a summary table containing sequence length, GC content, and base composition.

## Option B — CDS Extraction from GenBank
Search for the smallest genome size data from a public database, download a GenBank file and extract all CDS sequences into FASTA format.

## Option C — Mitochondrial Genome Comparison
Search for the four smallest genome size data from a public database, download mitochondrial genome sequences from multiple species and compare length and GC content.

## Option D — Alignment and Phylogenetics
Create a multiple sequence alignment and draw a phylogenetic tree.

## Option E — Livestock Gene Analysis
Retrieve sequences for a livestock gene from a public database of interest and perform sequence translation and comparison.